# Peterson & McCabe Developmental Analysis v1

**Goal:** Replicate ECSC Frog Story developmental effect using memory-curve / local-lag machinery on personal narratives.

**Dataset:** Peterson & McCabe corpus (CHILDES)
- 96 children, ages 4-9 years
- ~1,092 personal narratives (~11 per child)
- All female, typically developing

**Analyses:**
1. **Developmental replication** - Shape metrics by age, regression with covariates
2. **Length-matched robustness** - Confirm age effect survives length control
3. **Mixed effects model** - Account for repeated narratives per child
4. **Within-child stability** - ICC and within-child vs between-child distances

**Key Question:** Does the Frog Story developmental trend replicate in open personal narratives?

In [ ]:
# Install dependencies (Colab)
!pip install -q torch transformers accelerate bitsandbytes scipy statsmodels pingouin

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# For mixed effects and ICC
import statsmodels.api as sm
import statsmodels.formula.api as smf
try:
    import pingouin as pg
    HAS_PINGOUIN = True
except ImportError:
    HAS_PINGOUIN = False
    print("pingouin not available - ICC will use manual calculation")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# === EDIT THIS FOR YOUR COLAB SETUP ===
DRIVE_BASE = '/content/drive/MyDrive/LRTIA'  # Your Drive folder
# =====================================

class Config:
    # Model
    MODEL_NAME = "mistralai/Mistral-7B-v0.1"
    USE_4BIT = True  # Works on both T4 and A100
    
    # Paths (built from DRIVE_BASE)
    DATA_FILE = f'{DRIVE_BASE}/Data/petmcc_processed/transcripts.jsonl'
    OUTPUT_DIR = f'{DRIVE_BASE}/Results/petmcc'
    
    # Data filtering
    MIN_WORDS = 20  # Minimum words per narrative
    MIN_TOKENS = 40  # Minimum tokens for analysis (lowered for short corpus)
    
    # Target region for scoring
    TARGET_LEN = 30  # Score last 30 tokens (adjusted for short narratives)
    
    # Context lengths - SHORT REGIME for this corpus
    # Most narratives are 80-150 tokens, so we use smaller windows
    CONTEXT_LENGTHS = [8, 16, 24, 32, 48, 64, 96, 128]  # Max usable ~100 for typical narrative
    
    # Half-life computation
    MAX_CONTEXT_FIXED = 96  # Fixed max for comparable half-life (not 256!)
    BENEFIT_EPS = 0.3  # Min perplexity benefit for valid half-life
    
    # Baseline fluency: score on first N tokens with short context
    BASELINE_CONTEXT = 16
    BASELINE_TARGET_LEN = 20

config = Config()

# Create output directory
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
print(f"Config:")
print(f"  DATA_FILE: {config.DATA_FILE}")
print(f"  OUTPUT_DIR: {config.OUTPUT_DIR}")
print(f"  MIN_TOKENS: {config.MIN_TOKENS}")
print(f"  TARGET_LEN: {config.TARGET_LEN}")
print(f"  CONTEXT_LENGTHS: {config.CONTEXT_LENGTHS}")
print(f"  MAX_CONTEXT_FIXED: {config.MAX_CONTEXT_FIXED}")

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE (Colab only)
# ============================================================

try:
    from google.colab import drive
    import os
    # Check if already mounted
    if os.path.exists('/content/drive/MyDrive'):
        print("Drive already mounted!")
    else:
        drive.mount('/content/drive')
        print("Drive mounted!")
except ImportError:
    print("Not running on Colab - using local paths")
    # Override paths for local testing
    config.DATA_FILE = '../data/petmcc_processed/transcripts.jsonl'
    config.OUTPUT_DIR = '../results/petmcc'
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================

print(f"Loading {config.MODEL_NAME}...")

if config.USE_4BIT and device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_NAME,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
        trust_remote_code=True,
    )
    if device == "cpu":
        model = model.to(device)

tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
model.eval()
print("Model loaded!")

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

print(f"Loading data from: {config.DATA_FILE}")

if not os.path.exists(config.DATA_FILE):
    raise FileNotFoundError(
        f"Data file not found: {config.DATA_FILE}\n"
        f"Please upload transcripts.jsonl to your Drive at: {DRIVE_BASE}/data/petmcc_processed/"
    )

# Load and parse records
records = []
with open(config.DATA_FILE) as f:
    for line in f:
        record = json.loads(line)
        # Parse the nested population JSON
        pop = json.loads(record['population'])
        
        records.append({
            'doc_id': record['doc_id'],
            'child_id': record['author_id'],  # This is the child identifier
            'text': record['text'],
            'age_months': pop['age_months'],
            'sex': pop.get('sex', 'unknown'),
            'narrative_idx': pop.get('narrative_idx', 0),
            'word_count': len(record['text'].split()),
        })

df_all = pd.DataFrame(records)

# Verify schema
print(f"\nLoaded {len(df_all)} records")
print(f"\nSample record:")
print(df_all.iloc[0].to_dict())

In [ ]:
# ============================================================
# AGE BINNING FUNCTION
# ============================================================

def age_bin(age_months):
    """Bin age in months into year groups.
    
    4yo: 48-59 months
    5yo: 60-71 months
    6yo: 72-83 months
    7yo: 84-95 months
    8-9yo: 96-119 months (combined due to smaller N at 9)
    """
    if age_months < 60:
        return '4yo'
    elif age_months < 72:
        return '5yo'
    elif age_months < 84:
        return '6yo'
    elif age_months < 96:
        return '7yo'
    else:
        return '8-9yo'

# Apply age binning
df_all['age_group'] = df_all['age_months'].apply(age_bin)

# Tokenize all texts to get token counts
print("Tokenizing all narratives...")
df_all['n_tokens'] = df_all['text'].apply(lambda x: len(tokenizer.encode(x)))

print(f"\nToken count summary:")
print(df_all['n_tokens'].describe())

In [ ]:
# ============================================================
# PRE-RUN SANITY CHECKS
# ============================================================

print("=" * 70)
print("PRE-RUN SANITY CHECKS")
print("=" * 70)

# 1. Token length distributions by age bin
print("\n1. TOKEN LENGTH BY AGE GROUP (before filtering)")
print("-" * 50)
age_groups = ['4yo', '5yo', '6yo', '7yo', '8-9yo']
for ag in age_groups:
    ag_df = df_all[df_all['age_group'] == ag]
    n_total = len(ag_df)
    n_children = ag_df['child_id'].nunique()
    tokens_mean = ag_df['n_tokens'].mean()
    tokens_median = ag_df['n_tokens'].median()
    tokens_q25 = ag_df['n_tokens'].quantile(0.25)
    tokens_q75 = ag_df['n_tokens'].quantile(0.75)
    print(f"  {ag}: n={n_total} ({n_children} children), tokens: mean={tokens_mean:.0f}, median={tokens_median:.0f}, IQR=[{tokens_q25:.0f}, {tokens_q75:.0f}]")

# 2. Minimum token requirement for analysis
min_required = config.TARGET_LEN + min(config.CONTEXT_LENGTHS)
print(f"\n2. MINIMUM TOKENS REQUIRED FOR ANALYSIS")
print(f"   TARGET_LEN ({config.TARGET_LEN}) + min context ({min(config.CONTEXT_LENGTHS)}) = {min_required} tokens")

# 3. Inclusion rates by age group
print(f"\n3. INCLUSION RATES BY AGE GROUP (>= {config.MIN_TOKENS} tokens)")
print("-" * 50)
df = df_all[df_all['n_tokens'] >= config.MIN_TOKENS].copy()
for ag in age_groups:
    ag_all = len(df_all[df_all['age_group'] == ag])
    ag_kept = len(df[df['age_group'] == ag])
    pct = 100 * ag_kept / ag_all if ag_all > 0 else 0
    print(f"  {ag}: {ag_kept}/{ag_all} kept ({pct:.1f}%)")

print(f"\n  TOTAL: {len(df)}/{len(df_all)} narratives kept ({100*len(df)/len(df_all):.1f}%)")
print(f"  Unique children: {df['child_id'].nunique()}")

# 4. Check for selection bias
print(f"\n4. SELECTION BIAS CHECK")
print("-" * 50)
included_mean_age = df['age_months'].mean()
excluded_mean_age = df_all[~df_all.index.isin(df.index)]['age_months'].mean() if len(df) < len(df_all) else np.nan
print(f"  Mean age (included): {included_mean_age:.1f} months")
print(f"  Mean age (excluded): {excluded_mean_age:.1f} months")
if not np.isnan(excluded_mean_age) and abs(included_mean_age - excluded_mean_age) > 6:
    print("  WARNING: Large age difference between included/excluded - possible selection bias!")

In [ ]:
# ============================================================
# SPOT CHECK: ONE NARRATIVE
# ============================================================

print("\n5. SPOT CHECK: EXAMPLE NARRATIVE")
print("=" * 70)

# Pick a mid-length narrative
example = df[df['n_tokens'].between(80, 120)].iloc[0] if len(df[df['n_tokens'].between(80, 120)]) > 0 else df.iloc[0]

print(f"doc_id: {example['doc_id']}")
print(f"child_id: {example['child_id']}")
print(f"age: {example['age_months']} months ({example['age_group']})")
print(f"word_count: {example['word_count']}")
print(f"n_tokens: {example['n_tokens']}")

# Show what context windows will be used
max_ctx_available = example['n_tokens'] - config.TARGET_LEN
usable_contexts = [c for c in config.CONTEXT_LENGTHS if c <= max_ctx_available]
print(f"\nTarget region: last {config.TARGET_LEN} tokens")
print(f"Max context available: {max_ctx_available} tokens")
print(f"Usable context lengths: {usable_contexts}")

print(f"\nText (first 300 chars):")
print(example['text'][:300] + "..." if len(example['text']) > 300 else example['text'])

In [ ]:
# ============================================================
# CORE SCORING FUNCTIONS
# ============================================================

@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """
    Compute perplexity on tokens in [target_start, target_end).
    
    Args:
        token_ids: Full sequence of token IDs
        target_start: Start index of target region (inclusive)
        target_end: End index of target region (exclusive)
    
    Returns:
        (perplexity, mean_nll)
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    
    n_target_tokens = target_end - target_start - 1  # -1 because we predict next token
    if n_target_tokens <= 0:
        return float('inf'), float('inf')
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    total_loss = 0.0
    count = 0
    
    # Score each token in target region (predicting token i+1 from position i)
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        next_token = token_ids[i + 1]
        token_loss = -log_probs[next_token].item()
        total_loss += token_loss
        count += 1
    
    if count == 0:
        return float('inf'), float('inf')
    
    mean_nll = total_loss / count
    perplexity = np.exp(mean_nll)
    
    return perplexity, mean_nll


@torch.no_grad()
def compute_baseline_nll(token_ids):
    """
    Compute baseline NLL as fluency proxy.
    
    Uses a fixed short context (BASELINE_CONTEXT) to score a fixed region
    (BASELINE_TARGET_LEN tokens), giving a length-independent fluency measure.
    """
    min_len = config.BASELINE_CONTEXT + config.BASELINE_TARGET_LEN
    if len(token_ids) < min_len:
        return np.nan
    
    # Score tokens [BASELINE_CONTEXT : BASELINE_CONTEXT + BASELINE_TARGET_LEN]
    # with context [0 : BASELINE_CONTEXT + BASELINE_TARGET_LEN]
    end_pos = config.BASELINE_CONTEXT + config.BASELINE_TARGET_LEN
    tokens_to_use = token_ids[:end_pos]
    
    _, mean_nll = compute_perplexity_on_region(
        tokens_to_use, 
        target_start=config.BASELINE_CONTEXT,
        target_end=end_pos
    )
    
    return mean_nll

In [ ]:
# ============================================================
# MEMORY CURVE ANALYSIS FUNCTIONS
# ============================================================

def compute_cumulative_min(contexts, perplexities):
    """Convert to monotone non-increasing curve using cumulative minimum."""
    order = np.argsort(contexts)
    contexts = np.array(contexts)[order]
    perplexities = np.array(perplexities)[order]
    ppl_mon = np.minimum.accumulate(perplexities)
    return contexts, ppl_mon


def interpolate_perplexity(contexts, perplexities, target_ctx):
    """Linearly interpolate perplexity at a target context length."""
    if target_ctx <= contexts[0]:
        return perplexities[0]
    if target_ctx >= contexts[-1]:
        return perplexities[-1]
    
    for i in range(len(contexts) - 1):
        if contexts[i] <= target_ctx <= contexts[i+1]:
            frac = (target_ctx - contexts[i]) / (contexts[i+1] - contexts[i])
            return perplexities[i] + frac * (perplexities[i+1] - perplexities[i])
    
    return perplexities[-1]


def compute_shape_metrics(contexts, perplexities):
    """
    Compute shape metrics from memory curve.
    
    Uses cumulative-min envelope and MAX_CONTEXT_FIXED for comparability.
    """
    contexts, ppl_mon = compute_cumulative_min(contexts, perplexities)
    
    ppl_at_min = ppl_mon[0]
    ctx_min = contexts[0]
    max_ctx_available = contexts[-1]
    max_ctx_used = min(max_ctx_available, config.MAX_CONTEXT_FIXED)
    
    ppl_at_max = interpolate_perplexity(contexts, ppl_mon, max_ctx_used)
    total_benefit = ppl_at_min - ppl_at_max
    benefit_ok = total_benefit >= config.BENEFIT_EPS
    
    # Half-life: context length for 50% benefit
    half_life = np.nan
    if benefit_ok:
        target_ppl = ppl_at_min - 0.5 * total_benefit
        for i in range(len(ppl_mon) - 1):
            if contexts[i+1] > max_ctx_used:
                break
            if ppl_mon[i] >= target_ppl >= ppl_mon[i+1]:
                frac = (ppl_mon[i] - target_ppl) / (ppl_mon[i] - ppl_mon[i+1])
                half_life = contexts[i] + frac * (contexts[i+1] - contexts[i])
                break
    
    # Early slope (first 32 tokens)
    ppl_at_32 = interpolate_perplexity(contexts, ppl_mon, 32) if 32 <= max_ctx_available else np.nan
    early_slope = (ppl_at_min - ppl_at_32) / (32 - ctx_min) if (not np.isnan(ppl_at_32) and 32 > ctx_min) else np.nan
    
    # Early drop percentage
    early_drop = ppl_at_min - ppl_at_32 if not np.isnan(ppl_at_32) else np.nan
    early_drop_pct = 100 * early_drop / total_benefit if (benefit_ok and not np.isnan(early_drop)) else np.nan
    
    # Late benefit (48->96)
    ppl_at_48 = interpolate_perplexity(contexts, ppl_mon, 48) if 48 <= max_ctx_available else np.nan
    ppl_at_96 = interpolate_perplexity(contexts, ppl_mon, 96) if 96 <= max_ctx_available else np.nan
    late_benefit = ppl_at_48 - ppl_at_96 if (not np.isnan(ppl_at_48) and not np.isnan(ppl_at_96)) else np.nan
    late_benefit_pct = 100 * late_benefit / total_benefit if (benefit_ok and not np.isnan(late_benefit)) else np.nan
    
    return {
        'ppl_min_ctx': ppl_at_min,
        'ppl_max_ctx': ppl_at_max,
        'total_benefit': total_benefit,
        'benefit_ok': benefit_ok,
        'half_life': half_life,
        'early_slope': early_slope,
        'early_drop_pct': early_drop_pct,
        'late_benefit_pct': late_benefit_pct,
        'max_ctx_used': max_ctx_used,
    }

In [ ]:
def analyze_document(text, n_tokens):
    """
    Analyze a single document.
    
    Target region: last TARGET_LEN tokens
    Context: varies from min(CONTEXT_LENGTHS) up to max available
    
    Returns:
        baseline_nll: fluency proxy
        curve_results: list of {context_length, perplexity, nll}
        meta: dict with analysis metadata
    """
    full_tokens = tokenizer.encode(text)
    
    # Verify token count
    assert len(full_tokens) == n_tokens, f"Token count mismatch: {len(full_tokens)} vs {n_tokens}"
    
    if n_tokens < config.MIN_TOKENS:
        return None, [], {'skip_reason': 'too_short', 'n_tokens': n_tokens}
    
    # Target region: last TARGET_LEN tokens
    target_end = n_tokens
    target_start = max(0, n_tokens - config.TARGET_LEN)
    actual_target_len = target_end - target_start
    
    # Max context available before target
    max_context_available = target_start
    
    if max_context_available < min(config.CONTEXT_LENGTHS):
        return None, [], {'skip_reason': 'insufficient_context', 'n_tokens': n_tokens}
    
    # Baseline NLL (fluency proxy)
    baseline_nll = compute_baseline_nll(full_tokens)
    
    # Compute perplexity at each context length
    curve_results = []
    for ctx_len in config.CONTEXT_LENGTHS:
        if ctx_len > max_context_available:
            continue
        
        # Window: [target_start - ctx_len : target_end]
        window_start = target_start - ctx_len
        tokens_window = full_tokens[window_start:target_end]
        
        # Target region within window
        local_target_start = ctx_len  # Target starts after context
        local_target_end = len(tokens_window)
        
        ppl, nll = compute_perplexity_on_region(tokens_window, local_target_start, local_target_end)
        
        curve_results.append({
            'context_length': ctx_len,
            'perplexity': ppl,
            'nll': nll,
        })
    
    meta = {
        'n_tokens': n_tokens,
        'target_start': target_start,
        'target_end': target_end,
        'actual_target_len': actual_target_len,
        'max_context_available': max_context_available,
        'n_context_lengths_used': len(curve_results),
    }
    
    return baseline_nll, curve_results, meta

In [ ]:
# ============================================================
# RUN ANALYSIS ON ALL DOCUMENTS
# ============================================================

all_curves = []  # Raw curve data
doc_metrics = []  # Per-document metrics

skipped = {'too_short': 0, 'insufficient_context': 0, 'no_curves': 0}

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Analyzing"):
    baseline_nll, curve_results, meta = analyze_document(row['text'], row['n_tokens'])
    
    if baseline_nll is None:
        skipped[meta.get('skip_reason', 'unknown')] = skipped.get(meta.get('skip_reason', 'unknown'), 0) + 1
        continue
    
    if len(curve_results) < 3:
        skipped['no_curves'] += 1
        continue
    
    # Store raw curve data
    for r in curve_results:
        all_curves.append({
            'doc_id': row['doc_id'],
            'child_id': row['child_id'],
            'age_months': row['age_months'],
            'age_group': row['age_group'],
            'context_length': r['context_length'],
            'perplexity': r['perplexity'],
            'nll': r['nll'],
        })
    
    # Compute shape metrics
    contexts = [r['context_length'] for r in curve_results]
    perplexities = [r['perplexity'] for r in curve_results]
    shape = compute_shape_metrics(contexts, perplexities)
    
    doc_metrics.append({
        'doc_id': row['doc_id'],
        'child_id': row['child_id'],
        'narrative_idx': row['narrative_idx'],
        'age_months': row['age_months'],
        'age_group': row['age_group'],
        'word_count': row['word_count'],
        'n_tokens': row['n_tokens'],
        'baseline_nll': baseline_nll,
        'max_ctx_used': shape['max_ctx_used'],
        
        # Shape metrics
        'half_life': shape['half_life'],
        'early_slope': shape['early_slope'],
        'early_drop_pct': shape['early_drop_pct'],
        'late_benefit_pct': shape['late_benefit_pct'],
        'total_benefit': shape['total_benefit'],
        'ppl_min_ctx': shape['ppl_min_ctx'],
        'ppl_max_ctx': shape['ppl_max_ctx'],
    })

metrics_df = pd.DataFrame(doc_metrics)
curves_df = pd.DataFrame(all_curves)

print(f"\n" + "=" * 50)
print(f"ANALYSIS COMPLETE")
print(f"=" * 50)
print(f"Narratives analyzed: {len(metrics_df)}")
print(f"Unique children: {metrics_df['child_id'].nunique()}")
print(f"Skipped: {skipped}")
print(f"Valid half-life values: {metrics_df['half_life'].notna().sum()}")

In [ ]:
# ============================================================
# POST-RUN: VERIFY NO SELECTION BIAS
# ============================================================

print("\nFINAL SAMPLE BY AGE GROUP")
print("-" * 50)
for ag in age_groups:
    ag_df = metrics_df[metrics_df['age_group'] == ag]
    if len(ag_df) > 0:
        print(f"  {ag}: {len(ag_df)} narratives, {ag_df['child_id'].nunique()} children, " +
              f"mean_tokens={ag_df['n_tokens'].mean():.0f}")

---
## 1. Developmental Replication

Compute shape metrics by age bin and run regression with covariates.

In [ ]:
# ============================================================
# BOOTSTRAP CI FUNCTION
# ============================================================

def bootstrap_ci(values, n_bootstrap=1000, ci=0.95, statistic=np.mean):
    """Compute bootstrap CI for a statistic."""
    values = np.array(values)
    values = values[~np.isnan(values)]
    
    if len(values) < 3:
        return np.nan, np.nan, np.nan
    
    rng = np.random.default_rng(42)
    boot_stats = []
    
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_stats.append(statistic(sample))
    
    mean = statistic(values)
    alpha = 1 - ci
    ci_low = np.percentile(boot_stats, 100 * alpha / 2)
    ci_high = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    
    return mean, ci_low, ci_high

In [ ]:
# ============================================================
# SHAPE METRICS BY AGE GROUP
# ============================================================

print("SHAPE METRICS BY AGE GROUP")
print("=" * 70)

shape_metrics = ['half_life', 'early_slope', 'early_drop_pct', 'late_benefit_pct']

summary_rows = []
for ag in age_groups:
    ag_df = metrics_df[metrics_df['age_group'] == ag]
    if len(ag_df) == 0:
        continue
    
    result = {
        'age_group': ag, 
        'n_narratives': len(ag_df),
        'n_children': ag_df['child_id'].nunique(),
        'mean_n_tokens': ag_df['n_tokens'].mean(),
    }
    
    for metric in shape_metrics:
        mean, ci_low, ci_high = bootstrap_ci(ag_df[metric].values)
        result[f'{metric}_mean'] = mean
        result[f'{metric}_ci_low'] = ci_low
        result[f'{metric}_ci_high'] = ci_high
    
    summary_rows.append(result)

summary_df = pd.DataFrame(summary_rows)

print(f"\n{'Age':<8} {'N':>6} {'Kids':>5} {'Tokens':>7} {'Half-Life':>28}")
print("-" * 60)
for _, row in summary_df.iterrows():
    hl_str = f"{row['half_life_mean']:.1f} [{row['half_life_ci_low']:.1f}, {row['half_life_ci_high']:.1f}]"
    print(f"{row['age_group']:<8} {row['n_narratives']:>6} {row['n_children']:>5} {row['mean_n_tokens']:>7.0f} {hl_str:>28}")

In [ ]:
# ============================================================
# REGRESSION: SHAPE METRICS ~ AGE + COVARIATES
# ============================================================

print("\nREGRESSION ANALYSIS")
print("=" * 70)

valid_df = metrics_df.dropna(subset=['half_life', 'baseline_nll']).copy()
print(f"Valid observations: {len(valid_df)}")

# Standardize predictors
valid_df['age_months_z'] = (valid_df['age_months'] - valid_df['age_months'].mean()) / valid_df['age_months'].std()
valid_df['n_tokens_z'] = (valid_df['n_tokens'] - valid_df['n_tokens'].mean()) / valid_df['n_tokens'].std()
valid_df['baseline_nll_z'] = (valid_df['baseline_nll'] - valid_df['baseline_nll'].mean()) / valid_df['baseline_nll'].std()

print("\n1. Simple correlation: half_life ~ age_months")
rho, p_rho = spearmanr(valid_df['age_months'], valid_df['half_life'])
r, p_r = pearsonr(valid_df['age_months'], valid_df['half_life'])
print(f"   Spearman: rho={rho:.3f}, p={p_rho:.4f}")
print(f"   Pearson:  r={r:.3f}, p={p_r:.4f}")

print("\n2. OLS: half_life ~ age_months + n_tokens + baseline_nll")
model_ols = smf.ols('half_life ~ age_months_z + n_tokens_z + baseline_nll_z', data=valid_df).fit()
print(model_ols.summary().tables[1])

age_coef = model_ols.params['age_months_z']
age_p = model_ols.pvalues['age_months_z']
age_ci = model_ols.conf_int().loc['age_months_z']
print(f"\n   Age effect: beta={age_coef:.3f}, p={age_p:.4f}, 95% CI=[{age_ci[0]:.3f}, {age_ci[1]:.3f}]")

In [ ]:
# ============================================================
# ALL SHAPE METRICS REGRESSION
# ============================================================

print("\nAGE EFFECTS FOR ALL SHAPE METRICS")
print("=" * 90)

regression_results = []

for metric in shape_metrics:
    metric_df = metrics_df.dropna(subset=[metric, 'baseline_nll']).copy()
    if len(metric_df) < 30:
        continue
        
    metric_df['age_months_z'] = (metric_df['age_months'] - metric_df['age_months'].mean()) / metric_df['age_months'].std()
    metric_df['n_tokens_z'] = (metric_df['n_tokens'] - metric_df['n_tokens'].mean()) / metric_df['n_tokens'].std()
    metric_df['baseline_nll_z'] = (metric_df['baseline_nll'] - metric_df['baseline_nll'].mean()) / metric_df['baseline_nll'].std()
    
    model = smf.ols(f'{metric} ~ age_months_z + n_tokens_z + baseline_nll_z', data=metric_df).fit()
    rho, p_rho = spearmanr(metric_df['age_months'], metric_df[metric])
    
    regression_results.append({
        'metric': metric,
        'n': len(metric_df),
        'rho': rho,
        'p_simple': p_rho,
        'beta_age': model.params['age_months_z'],
        'p_age': model.pvalues['age_months_z'],
        'ci_low': model.conf_int().loc['age_months_z'][0],
        'ci_high': model.conf_int().loc['age_months_z'][1],
    })

reg_df = pd.DataFrame(regression_results)

print(f"\n{'Metric':<18} {'N':>5} {'rho':>7} {'p_simple':>9} {'beta':>8} {'p_ctrl':>8} {'95% CI':>22} {'Sig':>4}")
print("-" * 95)
for _, row in reg_df.iterrows():
    ci_str = f"[{row['ci_low']:.2f}, {row['ci_high']:.2f}]"
    sig = '*' if row['p_age'] < 0.05 else ''
    print(f"{row['metric']:<18} {row['n']:>5} {row['rho']:>7.3f} {row['p_simple']:>9.4f} {row['beta_age']:>8.3f} {row['p_age']:>8.4f} {ci_str:>22} {sig:>4}")

In [ ]:
# ============================================================
# PLOT: MEMORY CURVES BY AGE GROUP
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Plot 1: Memory curves
ax1 = axes[0]
for i, ag in enumerate(age_groups):
    ag_curves = curves_df[curves_df['age_group'] == ag]
    if len(ag_curves) == 0:
        continue
    curve_agg = ag_curves.groupby('context_length')['perplexity'].agg(['mean', 'std', 'count']).reset_index()
    curve_agg['se'] = curve_agg['std'] / np.sqrt(curve_agg['count'])
    n_docs = ag_curves['doc_id'].nunique()
    ax1.plot(curve_agg['context_length'], curve_agg['mean'], 
             color=colors[i], label=f'{ag} (n={n_docs})', linewidth=2)
    ax1.fill_between(curve_agg['context_length'], 
                     curve_agg['mean'] - 1.96*curve_agg['se'],
                     curve_agg['mean'] + 1.96*curve_agg['se'],
                     color=colors[i], alpha=0.2)

ax1.set_xlabel('Context Length (tokens)', fontsize=12)
ax1.set_ylabel('Perplexity', fontsize=12)
ax1.set_title('Memory Curves by Age Group', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Half-life by age
ax2 = axes[1]
x_pos = range(len(summary_df))
ax2.bar(x_pos, summary_df['half_life_mean'], 
        yerr=[summary_df['half_life_mean'] - summary_df['half_life_ci_low'],
              summary_df['half_life_ci_high'] - summary_df['half_life_mean']],
        capsize=5, color=colors[:len(summary_df)], alpha=0.7)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(summary_df['age_group'])
ax2.set_xlabel('Age Group', fontsize=12)
ax2.set_ylabel('Half-Life (tokens)', fontsize=12)
ax2.set_title('Half-Life by Age (95% CI)', fontsize=14)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/age_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Length-Matched Robustness

In [ ]:
# ============================================================
# LENGTH-MATCHED ANALYSIS
# ============================================================

print("LENGTH-MATCHED ROBUSTNESS")
print("=" * 70)

# Find overlapping token range
length_ranges = {}
for ag in age_groups:
    ag_tokens = valid_df[valid_df['age_group'] == ag]['n_tokens']
    if len(ag_tokens) >= 10:
        length_ranges[ag] = (ag_tokens.quantile(0.25), ag_tokens.quantile(0.75))
        print(f"{ag}: token Q1={length_ranges[ag][0]:.0f}, Q3={length_ranges[ag][1]:.0f}")

common_low = max(v[0] for v in length_ranges.values())
common_high = min(v[1] for v in length_ranges.values())
print(f"\nCommon band: {common_low:.0f} - {common_high:.0f} tokens")

matched_df = valid_df[(valid_df['n_tokens'] >= common_low) & (valid_df['n_tokens'] <= common_high)].copy()

print(f"\nLength-matched sample:")
for ag in age_groups:
    ag_df = matched_df[matched_df['age_group'] == ag]
    if len(ag_df) >= 5:
        print(f"  {ag}: n={len(ag_df)}, children={ag_df['child_id'].nunique()}")

if len(matched_df) >= 50:
    matched_df['age_months_z'] = (matched_df['age_months'] - matched_df['age_months'].mean()) / matched_df['age_months'].std()
    matched_df['n_tokens_z'] = (matched_df['n_tokens'] - matched_df['n_tokens'].mean()) / matched_df['n_tokens'].std()
    matched_df['baseline_nll_z'] = (matched_df['baseline_nll'] - matched_df['baseline_nll'].mean()) / matched_df['baseline_nll'].std()
    
    rho_m, p_rho_m = spearmanr(matched_df['age_months'], matched_df['half_life'])
    print(f"\nSimple correlation: rho={rho_m:.3f}, p={p_rho_m:.4f}")
    
    model_m = smf.ols('half_life ~ age_months_z + n_tokens_z + baseline_nll_z', data=matched_df).fit()
    age_coef_m = model_m.params['age_months_z']
    age_p_m = model_m.pvalues['age_months_z']
    age_ci_m = model_m.conf_int().loc['age_months_z']
    print(f"Controlled: beta={age_coef_m:.3f}, p={age_p_m:.4f}, CI=[{age_ci_m[0]:.3f}, {age_ci_m[1]:.3f}]")
    
    length_matched_survives = age_p_m < 0.05
    print(f"\n--> Age effect {'SURVIVES' if length_matched_survives else 'does NOT survive'} length matching")
else:
    print("Insufficient samples for length-matched analysis")
    length_matched_survives = None

---
## 3. Mixed Effects Model

In [ ]:
# ============================================================
# MIXED EFFECTS MODEL
# ============================================================

print("MIXED EFFECTS MODEL")
print("=" * 70)
print("Model: half_life ~ age_months + n_tokens + baseline_nll + (1|child_id)")

try:
    me_model = smf.mixedlm(
        'half_life ~ age_months_z + n_tokens_z + baseline_nll_z',
        data=valid_df,
        groups=valid_df['child_id']
    ).fit(method='powell')
    
    print("\nFixed Effects:")
    print(me_model.summary().tables[1])
    
    me_age_coef = me_model.fe_params['age_months_z']
    me_age_se = me_model.bse_fe['age_months_z']
    me_age_z = me_age_coef / me_age_se
    me_age_p = 2 * (1 - stats.norm.cdf(abs(me_age_z)))
    
    print(f"\nAge effect (mixed): beta={me_age_coef:.3f}, SE={me_age_se:.3f}, p={me_age_p:.4f}")
    
    var_child = me_model.cov_re.iloc[0, 0]
    var_resid = me_model.scale
    icc_model = var_child / (var_child + var_resid)
    
    print(f"\nVariance Components:")
    print(f"   Child: {var_child:.3f}")
    print(f"   Residual: {var_resid:.3f}")
    print(f"   ICC: {icc_model:.3f}")
    
except Exception as e:
    print(f"Mixed model failed: {e}")
    print("Using clustered SEs instead...")
    model_clust = smf.ols('half_life ~ age_months_z + n_tokens_z + baseline_nll_z', data=valid_df).fit(
        cov_type='cluster', cov_kwds={'groups': valid_df['child_id']}
    )
    print(model_clust.summary().tables[1])

---
## 4. Within-Child Stability

In [ ]:
# ============================================================
# WITHIN-CHILD ICC
# ============================================================

print("WITHIN-CHILD STABILITY")
print("=" * 70)

# Residualize for age/length/fluency
valid_df['half_life_resid'] = smf.ols(
    'half_life ~ age_months_z + n_tokens_z + baseline_nll_z', data=valid_df
).fit().resid

# Children with 2+ narratives
child_counts = valid_df.groupby('child_id').size()
multi_children = child_counts[child_counts >= 2].index
multi_df = valid_df[valid_df['child_id'].isin(multi_children)].copy()

print(f"Children with 2+ narratives: {len(multi_children)}")
print(f"Narratives from these: {len(multi_df)}")

def compute_icc_manual(df, target_col, group_col):
    groups = df[group_col].unique()
    k = len(groups)
    n_total = len(df)
    grand_mean = df[target_col].mean()
    
    n_per = df.groupby(group_col).size().values
    means = df.groupby(group_col)[target_col].mean()
    
    SS_b = sum(n * (m - grand_mean)**2 for n, m in zip(n_per, means))
    SS_w = sum((df[df[group_col] == g][target_col] - means[g]).pow(2).sum() for g in groups)
    
    MS_b = SS_b / (k - 1)
    MS_w = SS_w / (n_total - k)
    n_avg = n_total / k
    
    return (MS_b - MS_w) / (MS_b + (n_avg - 1) * MS_w)

if HAS_PINGOUIN and len(multi_df) >= 20:
    icc_raw = pg.intraclass_corr(data=multi_df, targets='doc_id', raters='child_id', ratings='half_life')
    icc_raw_val = icc_raw[icc_raw['Type'] == 'ICC1']['ICC'].values[0]
    icc_resid = pg.intraclass_corr(data=multi_df, targets='doc_id', raters='child_id', ratings='half_life_resid')
    icc_resid_val = icc_resid[icc_resid['Type'] == 'ICC1']['ICC'].values[0]
else:
    icc_raw_val = compute_icc_manual(multi_df, 'half_life', 'child_id')
    icc_resid_val = compute_icc_manual(multi_df, 'half_life_resid', 'child_id')

print(f"\nICC Results:")
print(f"   Raw half_life:         {icc_raw_val:.3f}")
print(f"   Residualized half_life: {icc_resid_val:.3f}")

if icc_resid_val > 0.3:
    print("   --> Moderate within-child stability (signature beyond development)")
elif icc_resid_val > 0.1:
    print("   --> Weak within-child stability")
else:
    print("   --> Little within-child stability")

In [ ]:
# ============================================================
# ICC FOR ALL METRICS
# ============================================================

print("\nICC FOR ALL SHAPE METRICS (residualized)")
print("-" * 50)

icc_results = []
for metric in shape_metrics:
    m_df = metrics_df.dropna(subset=[metric, 'baseline_nll']).copy()
    m_df['age_z'] = (m_df['age_months'] - m_df['age_months'].mean()) / m_df['age_months'].std()
    m_df['tok_z'] = (m_df['n_tokens'] - m_df['n_tokens'].mean()) / m_df['n_tokens'].std()
    m_df['nll_z'] = (m_df['baseline_nll'] - m_df['baseline_nll'].mean()) / m_df['baseline_nll'].std()
    m_df[f'{metric}_resid'] = smf.ols(f'{metric} ~ age_z + tok_z + nll_z', data=m_df).fit().resid
    
    counts = m_df.groupby('child_id').size()
    multi = counts[counts >= 2].index
    m_multi = m_df[m_df['child_id'].isin(multi)]
    
    if len(m_multi) >= 20:
        icc = compute_icc_manual(m_multi, f'{metric}_resid', 'child_id')
        icc_results.append({'metric': metric, 'icc': icc, 'n': len(m_multi), 'children': len(multi)})
        print(f"{metric:<18}: ICC={icc:.3f} (n={len(m_multi)}, {len(multi)} children)")

icc_df = pd.DataFrame(icc_results)

---
## 5. Summary & Deliverables

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================

print("\n" + "=" * 100)
print("SUMMARY TABLE")
print("=" * 100)

summary_table = []
for metric in shape_metrics:
    row = {'Metric': metric}
    
    r = reg_df[reg_df['metric'] == metric]
    if len(r) > 0:
        r = r.iloc[0]
        row['beta'] = f"{r['beta_age']:.3f}"
        row['p'] = f"{r['p_age']:.4f}"
        row['95% CI'] = f"[{r['ci_low']:.2f}, {r['ci_high']:.2f}]"
        row['Sig'] = 'Yes' if r['p_age'] < 0.05 else 'No'
    else:
        row['beta'] = row['p'] = row['95% CI'] = row['Sig'] = 'N/A'
    
    row['LengthMatched'] = 'Yes' if (metric == 'half_life' and length_matched_survives) else ('-' if metric != 'half_life' else 'No')
    
    i = icc_df[icc_df['metric'] == metric]
    row['ICC'] = f"{i.iloc[0]['icc']:.3f}" if len(i) > 0 else 'N/A'
    
    summary_table.append(row)

summary_table_df = pd.DataFrame(summary_table)
print(summary_table_df.to_string(index=False))

In [ ]:
# ============================================================
# FINAL PLOTS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Age curves
ax1 = axes[0]
for i, ag in enumerate(age_groups):
    ag_c = curves_df[curves_df['age_group'] == ag]
    if len(ag_c) > 0:
        agg = ag_c.groupby('context_length')['perplexity'].mean().reset_index()
        ax1.plot(agg['context_length'], agg['perplexity'], color=colors[i], label=ag, lw=2)
ax1.set_xlabel('Context')
ax1.set_ylabel('Perplexity')
ax1.set_title('Memory Curves')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Residualized scatter
ax2 = axes[1]
ax2.scatter(valid_df['age_months'], valid_df['half_life_resid'], alpha=0.3, s=15)
ax2.axhline(0, color='gray', ls='--', alpha=0.5)
ax2.set_xlabel('Age (months)')
ax2.set_ylabel('Half-Life (resid)')
ax2.set_title('Age Effect (controlled)')
ax2.grid(alpha=0.3)

# 3. ICC bar
ax3 = axes[2]
if len(icc_df) > 0:
    ax3.bar(range(len(icc_df)), icc_df['icc'], color='steelblue', alpha=0.7)
    ax3.set_xticks(range(len(icc_df)))
    ax3.set_xticklabels(icc_df['metric'], rotation=45, ha='right')
    ax3.set_ylabel('ICC')
    ax3.set_title('Within-Child Stability')
    ax3.axhline(0, color='gray', ls='--', alpha=0.5)
    ax3.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/summary_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# INTERPRETATION
# ============================================================

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)

hl = reg_df[reg_df['metric'] == 'half_life'].iloc[0]
sig = hl['p_age'] < 0.05
direction = 'increases' if hl['beta_age'] > 0 else 'decreases'

print(f"\n1. DEVELOPMENTAL EFFECT")
if sig:
    print(f"   Half-life significantly {direction} with age (p={hl['p_age']:.4f})")
    print(f"   Effect: beta={hl['beta_age']:.3f} (standardized)")
else:
    print(f"   No significant age effect (p={hl['p_age']:.4f})")

print(f"\n2. ROBUSTNESS")
if length_matched_survives is not None:
    print(f"   Length-matched: {'SURVIVES' if length_matched_survives else 'does NOT survive'}")

print(f"\n3. WITHIN-CHILD STABILITY")
print(f"   ICC (residualized): {icc_resid_val:.3f}")

print(f"\n4. COMPARISON TO FROG STORY")
if sig:
    print("   --> REPLICATES developmental trend from Frog Story")
else:
    print("   --> Does NOT replicate - effect may be task-specific")

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

metrics_df.to_csv(f'{config.OUTPUT_DIR}/doc_metrics.csv', index=False)
summary_df.to_csv(f'{config.OUTPUT_DIR}/age_summary.csv', index=False)
reg_df.to_csv(f'{config.OUTPUT_DIR}/regression.csv', index=False)
if len(icc_df) > 0:
    icc_df.to_csv(f'{config.OUTPUT_DIR}/icc.csv', index=False)
summary_table_df.to_csv(f'{config.OUTPUT_DIR}/summary_table.csv', index=False)

print(f"Results saved to: {config.OUTPUT_DIR}")